# Nexus N1 — Segment-Native World Model (Strict Bottleneck)

One-click: clone the repo, train the **full N1 world model** (`seg=scheduled`) on Crafter on the GPU, and stream checkpoints + `metrics.jsonl` to Google Drive.

N1 is **WM-only** (no actor yet — that's N2). It answers the bottleneck bet: does `(u_n, z_{t_n})` suffice to re-init prediction across a boundary? The deliverable is the **post-boundary NLL spike** per `(w, G)` cell.

**Before running:** Runtime → Change runtime type → **GPU** (A100 / L4 / V100 all fine on Colab Pro).

Start with the two corners of the §11.4 grid: **w=0/G=4** (strict reference) and **w=64/G=4** (leak reference).

In [ ]:
# 1. Confirm the GPU
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2. Get the code (nexus imports emerald_torch as a parts library; only needs crafter)
%cd /content
!git clone https://github.com/JuliusSamwer/Nexus_v1.git 2>/dev/null || (cd Nexus_v1 && git pull -q)
%cd /content/Nexus_v1
!pip install -q crafter
print('ready')

In [ ]:
# 3. Mount Drive so checkpoints + metrics survive disconnects
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/nexus'
os.makedirs(DRIVE, exist_ok=True)
print('artifacts ->', DRIVE)

In [ ]:
# 4. ---- the run knobs ----
W = 0          # leak width: 0 (strict) | 16 | 64
G = 4          # slow tokens per segment: 2 | 4 | 8
STEPS = 50000  # env steps (~1.5-3h on Colab Pro; bump to 100000 for a full N1 run)

LOGDIR = f'{DRIVE}/n1_w{W}_G{G}'
os.makedirs(LOGDIR, exist_ok=True)
print('logdir:', LOGDIR)

In [ ]:
# 5. Train. Streams loss + the §8 diagnostics (post_boundary_spike, slow_advantage, u_ppl).
!python3 -m nexus.train --configs crafter --device cuda --logdir "{LOGDIR}" \
    --w {W} --G {G} --steps {STEPS}

In [ ]:
# 6. Watch the N1 diagnostics (read metrics.jsonl; re-run anytime, even mid-training)
import json, glob
import matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(os.path.join(LOGDIR, 'metrics.jsonl')) if l.strip()]
rows = [r for r in rows if 'loss' in r]
step = [r['step'] for r in rows]
def g(k):
    return [r.get(k, float('nan')) for r in rows]
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
panels = [
    ('fast_image', 'fast decoder NLL (recon)'),
    ('post_boundary_spike', 'post-boundary spike  (§8.1 HEADLINE)'),
    ('slow_advantage', 'slow advantage = copy - ground  (§8.2; want > 0)'),
    ('ground_nll_diag', 'grounding NLL (next boundary frame)'),
    ('u_perplexity', 'u perplexity  (§8.4 collapse alarm)'),
    ('fast_kl_prior', 'fast prior KL'),
]
for a, (k, title) in zip(ax.flat, panels):
    a.plot(step, g(k)); a.set_title(title); a.set_xlabel('env steps'); a.grid(alpha=0.3)
    if k in ('post_boundary_spike', 'slow_advantage'):
        a.axhline(0, color='r', ls='--', lw=0.8)
plt.tight_layout(); plt.show()
last = rows[-1]
print(f"@ step {last['step']}:  spike={last.get('post_boundary_spike'):.3f}  "
      f"slow_adv={last.get('slow_advantage'):.2f}  u_ppl={last.get('u_perplexity'):.1f}")
print('per-offset fast NLL:', {f'off{k}': round(last.get(f'fast_nll_off{k}', float('nan')), 2) for k in range(1,6)},
      '| mid', round(last.get('fast_nll_mid', float('nan')), 2))

## Run the grid

Change `W` / `G` in **cell 4** and re-run cells 4–6 for each cell of `{w∈ 0,16,64} × {G∈ 2,4,8}`. Start with the corners: **w=0/G=4** then **w=64/G=4**. Each writes its own `n1_w{W}_G{G}/` on Drive.

**How to read it (§8):**
- **`post_boundary_spike`** (fast-prior NLL at offset 1 minus mid-segment): small at `w=0` ⇒ strictness is free, `(u, z_b)` suffices. Shrinks as `w` grows ⇒ unobservable environmental memory genuinely needed — *that's the headline figure*. Stays large at `w=64` ⇒ an Init/`G` problem, not a leak problem (F1).
- **`slow_advantage`** should climb **positive** (grounding beats copy-last-boundary-frame) — the slow tier earning its existence.
- **`u_perplexity`** well above 1 (out of `u_classes=256`) = no codebook collapse.

Training is resumable-safe in the sense that `latest.pt` is on Drive; re-running cell 5 restarts from scratch (N1 has no resume-from-checkpoint yet — ask if you want it).